# Platform-agnostic training notebook

This notebook is designed to work on Kaggle, Google Colab, RunPod, and local GPU/CPU machines without depending on uv or Windows-specific commands.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path


def resolve_repo_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd.parent,
        cwd / "lux-llm-factory",
        cwd.parent / "lux-llm-factory",
        cwd / "LuxLLMFactory",
        cwd.parent / "LuxLLMFactory",
        Path("/kaggle/working/LuxLLMFactory"),
        Path("/kaggle/input/LuxLLMFactory"),
        Path("/kaggle/working/lux-llm-factory"),
        Path("/kaggle/input/lux-llm-factory"),
        Path("/kaggle/input/datasets/markzither/luxllmfactory"),
    ]

    for candidate in candidates:
        if (candidate / ".git").exists() or (candidate / "pyproject.toml").exists():
            return candidate

    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        for base in [Path("/kaggle/working"), Path("/kaggle/input")]:
            for candidate in sorted(base.glob("*")):
                if (candidate / "pyproject.toml").exists() or (candidate / ".git").exists():
                    return candidate

    return cwd.parent if cwd.name == "notebooks" else cwd


REPO_DIR = resolve_repo_dir()
print("Working root:", Path.cwd().resolve())
print("Repo dir:", REPO_DIR)

if not REPO_DIR.exists():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        raise FileNotFoundError(
            "The repo was not found under the expected Kaggle locations. "
            "Please ensure your uploaded dataset folder contains pyproject.toml or .git and is visible in /kaggle/working or /kaggle/input."
        )
    raise FileNotFoundError("Repository directory not found. Please run this notebook from the repo root or upload the repo contents.")

os.chdir(REPO_DIR)
print("Repository ready at", REPO_DIR)

Working root: C:\Source\github\lod-mcp\lux-llm-factory\notebooks
Repo dir: C:\Source\github\lod-mcp\lux-llm-factory
Repository ready at C:\Source\github\lod-mcp\lux-llm-factory


In [3]:
import os
import subprocess
import sys
from pathlib import Path

packages = [
    "pip",
    "torch>=2.4",
    "transformers>=4.44",
    "datasets>=2.20",
    "peft>=0.12",
    "trl>=0.10.1",
    "accelerate>=0.33",
    "evaluate>=0.4.1,<0.5",
    "pyyaml>=6",
    "sentencepiece>=0.2",
]

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", *packages], check=True)

print("Dependencies installed.")

Dependencies installed.


In [4]:
import os
import platform

platform_name = "unknown"
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
    platform_name = "kaggle"
elif "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ:
    platform_name = "colab"
elif os.environ.get("RUNPOD_POD_ID"):
    platform_name = "runpod"
else:
    platform_name = "local"

print("Detected platform:", platform_name)
print("Python:", platform.python_version())

Detected platform: local
Python: 3.14.1


In [6]:
import os
import subprocess
import sys

config_path = os.environ.get("TRAIN_CONFIG", "training/configs/first-run-cpu.yaml")

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

print("Using config:", config_path)


def run_streaming_command(command, cwd=None):
    print("$", " ".join(command))
    process = subprocess.Popen(
        command,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


run_streaming_command([sys.executable, "scripts/train_first_sft.py", "--config", config_path], cwd=os.getcwd())

Using config: training/configs/first-run-cpu.yaml
$ c:\Source\github\lod-mcp\lux-llm-factory\.venv\Scripts\python.exe scripts/train_first_sft.py --config training/configs/first-run-cpu.yaml

Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 29/29 [00:00<00:00, 6401.50it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sshleifer/tiny-gpt2
Key                                   | Status     |  | 
--------------------------------------+------------+--+-
transformer.h.{0, 1}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
c:\Source\github\lod-mcp\lux-llm-factory\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and ge